# 02 — Diagnostics: why are we 0.11 behind the leader?

Current best: **0.829085** (50/50 blend). Leaderboard leader: **0.938827**.

A gap that size is not closed by a bigger backbone. Two hypotheses, both testable
here in ~10 minutes, **with no training**:

| Check | Hypothesis | If true |
|---|---|---|
| **1** | Training labels are noisy and the model is learning the mistakes | clean them, retrain (~45 min) |
| **2** | Test frames have near-duplicate twins in train (1 FPS video) | copy labels, no retraining at all |

Both measure themselves on **validation data first**. Nothing is applied to the test
set unless the check earns it.

**Nothing here overwrites your checkpoints, your OOF, or your 0.829 submission.**

⚠️ **Manual decisions are marked 🔶 — everything else is automatic.**

## 1. Setup

In [ ]:
!pip install -q timm albumentations

import os, sys, subprocess
os.chdir('/content')
REPO_DIR = '/content/OctWave3'

try:
    from google.colab import userdata
    TOKEN = userdata.get('GH_TOKEN')
except Exception:
    TOKEN = None
URL = (f'https://{TOKEN}@github.com/sasindu345/OctWave3.git' if TOKEN
       else 'https://github.com/sasindu345/OctWave3.git')

def run(*a, cwd=None):
    r = subprocess.run(a, cwd=cwd, capture_output=True, text=True)
    m = (r.stdout + r.stderr).strip()
    if TOKEN: m = m.replace(TOKEN, '***')
    if m: print(m)
    return r.returncode

if os.path.isdir(REPO_DIR + '/.git'):
    run('git', 'pull', '-q', URL, 'main', cwd=REPO_DIR)
else:
    run('git', 'clone', '-q', URL, REPO_DIR)

os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
import importlib; importlib.invalidate_caches()

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/octwave3/outputs')

# data (skips if already present)
if not os.path.isdir('/content/data/images'):
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files; files.upload()
        !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle competitions download -c oct-wave-3-0-kaggle-challenge-02 -p /content/data
    !unzip -q -o '/content/data/*.zip' -d /content/data
    !if [ -f /content/data/images.zip ]; then unzip -q -o /content/data/images.zip -d /content/data; fi
print('setup done')

## 2. Config — points at the models you already trained

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from pathlib import Path
for m in ('src.config','src.utils','src.dataset','src.model','src.predict','src.analysis','src.diagnostics'):
    importlib.reload(importlib.import_module(m))
from src.config import cfg

cfg.data_dir = Path('/content/data')
cfg.out_dir  = DRIVE_OUT
cfg.exp_name = 'exp05_multilabel_b0'     # the OOF we diagnose
cfg.head     = 'multilabel'
cfg.num_classes = 4
FOLDS = [0, 1, 2, 3, 4]

from src.dataset import build_dataframe, build_test_dataframe
train_df = build_dataframe(cfg)
test_df  = build_test_dataframe(cfg)
print(f'train {len(train_df)} | test {len(test_df)}')

## 3. CHECK 1 — Are the training labels wrong?

The competition says training labels are **noisy** and test labels are **clean**.
This scans your out-of-fold predictions for images where the model is *confident*
the label is wrong. Because the predictions are out-of-fold, the model never saw
those images in training — so a confident disagreement is evidence about the label,
not memorisation.

Read the `pct_of_train` column: how much of your training set looks mislabeled.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from src.diagnostics import label_noise_report
noise = label_noise_report(cfg.out_dir, cfg.exp_name, FOLDS, train_df)

## 4. CHECK 2 — Do test frames have twins in the training set?

Images are video frames at 1 FPS, split randomly between train and test — so frames
one second apart may sit on opposite sides of the split while looking nearly identical.

For every **validation** image we find its nearest **training** image and ask whether
their labels agree, bucketed by similarity. If agreement is very high in the top
band, matching beats predicting.

Embeddings use **plain ImageNet weights, not your fine-tuned model** — a fine-tuned
model has seen these images and would embed them distinctively, inflating exactly
the similarities we are trying to measure.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import numpy as np, copy
from src.diagnostics import extract_embeddings, duplicate_report

emb_cfg = copy.deepcopy(cfg)
emb_cfg.head = 'softmax'          # generic backbone, head is discarded anyway
emb_cfg.model_name = 'tf_efficientnet_b0'
emb_cfg.pretrained = True

emb_train = extract_embeddings(emb_cfg, None, train_df)      # None = ImageNet weights
np.save(cfg.out_dir / 'emb_train.npy', emb_train)

dup = duplicate_report(emb_train, train_df, FOLDS)

## 5. 🔶 DECISION POINT — read the two verdicts

| Check 2 verdict | Check 1 verdict | Do this |
|---|---|---|
| `ACT` | anything | **Section 6** — apply neighbour labels, no retraining |
| `SKIP` | `ACT` | **Section 7** — write cleaned labels, then retrain in notebook 01 |
| `SKIP` | `MARGINAL` / `SKIP` | **Section 8** — neither hypothesis holds, fall back to more models |

Run the cell below; it prints which section to go to.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

print(f"CHECK 1 (label noise) : {noise['verdict']:9s} - {noise['why']}")
print(f"CHECK 2 (duplicates)  : {dup['verdict']:9s} - {dup['why']}")
print()
if dup['verdict'] == 'ACT':
    print('>>> GO TO SECTION 6 (neighbour labels - no retraining needed)')
elif noise['verdict'] == 'ACT':
    print('>>> GO TO SECTION 7 (clean labels, then retrain)')
else:
    print('>>> GO TO SECTION 8 (neither holds - train another architecture instead)')

## 6. Apply neighbour labels — only if CHECK 2 said `ACT`

Two steps, and the first one is a dress rehearsal on validation data.

**6a** measures what the rule would do to macro F1 on OOF, per fold. If it does not
improve on 4+ folds by more than the 0.018 noise floor, **stop** — do not run 6b.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

# 6a. REHEARSAL on validation - decides whether 6b is allowed to run
import numpy as np
from sklearn.metrics import f1_score
from src.diagnostics import _topk_neighbours
from src.utils import load_oof

SIM_THRESHOLD = 0.95      # 🔶 set from the CHECK 2 table: highest band with agreement >= 0.95

y = train_df.target.values
deltas = []
for f in FOLDS:
    va = np.where(train_df.fold.values == f)[0]
    tr = np.where(train_df.fold.values != f)[0]
    p, t = load_oof(cfg.out_dir, cfg.exp_name, f)

    sims, idxs = _topk_neighbours(emb_train[va], emb_train[tr], k=1)
    sim, nn_lab = sims[:, 0], y[tr[idxs[:, 0]]]

    before = f1_score(t, p.argmax(1), average='macro')
    pred = p.argmax(1).copy()
    m = sim >= SIM_THRESHOLD
    pred[m] = nn_lab[m]
    after = f1_score(t, pred, average='macro')
    deltas.append(after - before)
    print(f'fold {f}: {before:.4f} -> {after:.4f} ({after-before:+.4f})  '
          f'overrode {m.sum()}/{len(va)}')

deltas = np.array(deltas)
print(f'\nmean {deltas.mean():+.4f} | folds improved {int((deltas>0).sum())}/5')
OK = deltas.mean() > 0.018 and (deltas > 0).sum() >= 4
print('VERDICT:', 'ADOPT - run 6b' if OK else 'REJECT - do NOT run 6b, go to section 7 or 8')

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

# 6b. Apply to TEST and download one CSV. Only run if 6a said ADOPT.
import numpy as np
from src.diagnostics import extract_embeddings, apply_neighbour_labels
from src.predict import make_submission

assert OK, '6a said REJECT - do not run this cell'

emb_test = extract_embeddings(emb_cfg, None, test_df)
# rebuild the 0.829 blend from the two saved probability files
champ = np.load(cfg.out_dir / 'exp02_5fold_b0_test_probs.npy')
ml    = np.load(cfg.out_dir / 'exp05_multilabel_b0_test_probs.npy')
blend = 0.5 * champ + 0.5 * ml      # the weights that scored 0.829085

adj, n = apply_neighbour_labels(blend, emb_test, emb_train,
                                train_df.target.values, SIM_THRESHOLD)
tag = f'nn{int(SIM_THRESHOLD*100)}_blend'
sub = make_submission(cfg, adj, test_df, f'{tag}.csv')
from google.colab import files
print(f'\n>>> THE ONE FILE TO UPLOAD: {tag}.csv <<<')
files.download(str(cfg.out_dir / 'submissions' / f'{tag}.csv'))

## 7. Clean the labels — only if CHECK 1 said `ACT` and CHECK 2 said `SKIP`

Writes `train_clean.csv` with the confidently-disputed rows **removed**. Removing is
safer than relabelling: relabelling teaches the model its own opinion, which
reinforces whatever it already gets wrong.

Then retrain in notebook 01 with `cfg.train_csv = 'train_clean.csv'` and a new
`exp_name`, and compare with `decide()` as usual.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import pandas as pd

CONF = 0.90       # 🔶 confidence above which a disagreement counts as a bad label
s = noise['suspects']
drop_idx = s.index[(s.model_pred != s.label) & (s.confidence >= CONF)]

raw = pd.read_csv(cfg.data_dir / 'train.csv')
clean = raw.drop(index=drop_idx).reset_index(drop=True)
out = cfg.data_dir / 'train_clean.csv'
clean.to_csv(out, index=False)

print(f'removed {len(drop_idx)} of {len(raw)} rows ({100*len(drop_idx)/len(raw):.1f}%)')
print('remaining class balance:\n', clean.appearance.value_counts().sort_index())
print(f'\nwrote {out}')
print('\nNEXT: notebook 01 -> cfg.train_csv = "train_clean.csv", '
      'cfg.exp_name = "exp07_cleaned", retrain 5 folds, then compare with decide().')

import shutil
shutil.copy(out, cfg.out_dir / 'train_clean.csv')   # keep a copy in Drive

## 8. Neither hypothesis holds

Then the gap is model quality, and the remaining levers are ordinary ones. In
notebook 01, in this order:

1. `EXPERIMENT = 'b2_300'` — different architecture *and* resolution (~70 min)
2. Blend champion + exp05 + b2 three ways
3. `EXPERIMENT = 'b0_repeat'` (seed 43) — cheap extra ensemble member

Each is worth roughly 0.01–0.02. They will not reach 0.93, but they compound.

## 9. Push results to GitHub

In [ ]:
import shutil, subprocess
from pathlib import Path
from google.colab import userdata
REPO = Path('/content/OctWave3')
(REPO / 'results' / 'figures').mkdir(parents=True, exist_ok=True)
log = DRIVE_OUT / 'run_log.jsonl'
if log.exists():
    shutil.copy(log, REPO / 'results' / 'run_log.jsonl')
TOKEN = userdata.get('GH_TOKEN')
def git(*a, secret=False):
    r = subprocess.run(('git',)+a, cwd=REPO, capture_output=True, text=True)
    m = (r.stdout + r.stderr).strip()
    if secret and TOKEN: m = m.replace(TOKEN, '***')
    if m: print(m)
    return r.returncode
git('config','user.email','cryptxgmora@gmail.com'); git('config','user.name','sasindu345')
git('add','results','logs')
if git('diff','--cached','--quiet') == 0:
    print('nothing new')
else:
    git('commit','-m','diagnostics run')
    git('push', f'https://{TOKEN}@github.com/sasindu345/OctWave3.git','HEAD:main', secret=True)